# Parameter Exploration — Lambda & p_fresh

Workspace for developing better estimators for `compute_edge()`'s two estimated parameters:
- **lambda_rate**: review arrival rate (reviews/hr)
- **p_fresh**: probability a future review is positive

The goal: build functions that take a movie's review history and return better estimates
than the naive defaults (`reviews_in_last_6h / 6` and `fresh/total`).

See `BACKLOG.md` §3 for context and `brainstorm/` for approach ideas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from scipy.stats import binom, poisson
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10,
})

ROOT = Path('..').resolve()

# Import compute_edge so you can test estimators end-to-end
from rotten_tomatoes_forecasting import compute_edge

print(f"Project root: {ROOT}")

## 1. Load data

In [ ]:
# ── Movies index ──────────────────────────────────────────────────────
mi = pd.read_csv(ROOT / 'movies_index.csv')

mi['volume'] = mi['Trading Volume ($)'].str.replace(r'[\$,]', '', regex=True).astype(float)
for col in ['Embargo Lift Date', 'Bet Open Date', 'Bet Close Date']:
    mi[col] = pd.to_datetime(mi[col], utc=True)

mi[['score_low', 'score_high']] = mi['Tomatometer Score Range Bet Close'].str.split('-', expand=True).astype(float)
mi['score_mid'] = (mi['score_low'] + mi['score_high']) / 2
mi['score_mid_pct'] = mi['score_mid'] * 100

mi[['reviews_low', 'reviews_high']] = mi['Total Reviews Bet Close'].str.split('-', expand=True).astype(float)
mi['reviews_mid'] = (mi['reviews_low'] + mi['reviews_high']) / 2

mi['embargo_to_close_days'] = (mi['Bet Close Date'] - mi['Embargo Lift Date']).dt.days

print(f"Movies: {len(mi)}")
mi[['Slug', 'volume', 'Bet Close Date', 'score_mid_pct', 'reviews_mid']].head(3)

In [ ]:
# ── Reviews ───────────────────────────────────────────────────────────
reviews = pd.read_csv(ROOT / 'reviews.csv')
reviews['estimated_timestamp'] = pd.to_datetime(reviews['estimated_timestamp'], utc=True, format='ISO8601')
reviews['scrape_time'] = pd.to_datetime(reviews['scrape_time'], utc=True, format='ISO8601')
reviews['is_fresh'] = reviews['tomatometer_sentiment'] == 'positive'

# Join bet close date onto reviews for hours_before_close calculations
close_map = mi.set_index('Slug')['Bet Close Date']
reviews['bet_close'] = reviews['movie_slug'].map(close_map)
reviews['hours_before_close'] = (reviews['bet_close'] - reviews['estimated_timestamp']).dt.total_seconds() / 3600
reviews['days_before_close'] = reviews['hours_before_close'] / 24

print(f"Reviews: {len(reviews):,}")
print(f"Movies in reviews: {reviews['movie_slug'].nunique()}")
print(f"Timestamp confidence: {reviews['timestamp_confidence'].value_counts().to_dict()}")
print(f"Top critics: {reviews['top_critic'].sum():,} ({reviews['top_critic'].mean()*100:.1f}%)")
print(f"Overall fresh rate: {reviews['is_fresh'].mean()*100:.1f}%")

In [ ]:
# ── Per-movie summary (pre-joined, useful throughout) ─────────────────
movie_stats = reviews.groupby('movie_slug').agg(
    n_reviews=('id', 'count'),
    n_fresh=('is_fresh', 'sum'),
    fresh_rate=('is_fresh', 'mean'),
    n_top=('top_critic', 'sum'),
    top_fresh_rate=('is_fresh', lambda s: s[reviews.loc[s.index, 'top_critic']].mean() if reviews.loc[s.index, 'top_critic'].any() else np.nan),
    nontop_fresh_rate=('is_fresh', lambda s: s[~reviews.loc[s.index, 'top_critic']].mean() if (~reviews.loc[s.index, 'top_critic']).any() else np.nan),
    earliest=('estimated_timestamp', 'min'),
    latest=('estimated_timestamp', 'max'),
    pct_minute=('timestamp_confidence', lambda s: (s == 'm').mean()),
    pct_hour=('timestamp_confidence', lambda s: (s == 'h').mean()),
).reset_index()
movie_stats = movie_stats.merge(mi[['Slug', 'Bet Close Date', 'Embargo Lift Date', 'volume', 'score_mid_pct', 'reviews_mid']], 
                                 left_on='movie_slug', right_on='Slug', how='left').drop(columns='Slug')
movie_stats['top_frac'] = movie_stats['n_top'] / movie_stats['n_reviews']
movie_stats['top_bias_pp'] = (movie_stats['top_fresh_rate'] - movie_stats['nontop_fresh_rate']) * 100

print(f"{len(movie_stats)} movies")
print(f"Minute-level data: {(movie_stats['pct_minute'] > 0.1).sum()} movies have >10% minute-level timestamps")
movie_stats[['movie_slug', 'n_reviews', 'fresh_rate', 'top_frac', 'top_bias_pp', 'pct_minute']].describe().round(2)

## 2. Helpers

Slice-and-dice functions for exploring reviews at different time snapshots. These simulate
"what would we have known at time T before close?" for any movie.

In [ ]:
def snapshot_at(slug: str, hours_before_close: float) -> dict:
    """Simulate the review state for a movie at T hours before close.
    
    Returns the counts/rates we would have known at that moment,
    plus what actually happened after (for validation).
    """
    mv = reviews[reviews['movie_slug'] == slug].copy()
    if mv.empty:
        raise ValueError(f"No reviews for {slug}")
    
    known = mv[mv['hours_before_close'] >= hours_before_close]
    future = mv[mv['hours_before_close'] < hours_before_close]
    
    fresh = known['is_fresh'].sum()
    total = len(known)
    top_known = known[known['top_critic']]
    nontop_known = known[~known['top_critic']]
    
    return {
        'slug': slug,
        'hours_before_close': hours_before_close,
        # What we know at time T
        'fresh': int(fresh),
        'total': int(total),
        'p_fresh_naive': fresh / total if total > 0 else np.nan,
        'top_fresh': int(top_known['is_fresh'].sum()),
        'top_total': int(len(top_known)),
        'nontop_fresh': int(nontop_known['is_fresh'].sum()),
        'nontop_total': int(len(nontop_known)),
        'top_frac': len(top_known) / total if total > 0 else np.nan,
        # What actually happens after (ground truth for validation)
        'future_reviews': int(len(future)),
        'future_fresh': int(future['is_fresh'].sum()),
        'future_fresh_rate': future['is_fresh'].mean() if len(future) > 0 else np.nan,
        'future_top_frac': future['top_critic'].mean() if len(future) > 0 else np.nan,
        'final_fresh': int(mv['is_fresh'].sum()),
        'final_total': int(len(mv)),
    }


def snapshot_all_movies(hours_before_close: float) -> pd.DataFrame:
    """Run snapshot_at for every movie at the same hours_before_close."""
    rows = []
    for slug in reviews['movie_slug'].unique():
        try:
            rows.append(snapshot_at(slug, hours_before_close))
        except ValueError:
            continue
    return pd.DataFrame(rows)


# Quick test
s = snapshot_at('forbidden_fruits_2026', 48)
print(f"forbidden_fruits at T-48h: {s['fresh']}/{s['total']} known, {s['future_reviews']} to come, "
      f"naive p_fresh={s['p_fresh_naive']:.3f}, actual future fresh rate={s['future_fresh_rate']:.3f}")

In [ ]:
def arrival_curve(slug: str, day_buckets=None) -> pd.DataFrame:
    """Build the review arrival curve for one movie.
    
    Returns a DataFrame with one row per day-bucket, showing cumulative
    and incremental review counts. Default buckets are whole days before close.
    """
    if day_buckets is None:
        day_buckets = list(range(0, 31))
    
    mv = reviews[reviews['movie_slug'] == slug].copy()
    if mv.empty:
        raise ValueError(f"No reviews for {slug}")
    
    total = len(mv)
    rows = []
    for d in sorted(day_buckets):
        arrived = mv[mv['days_before_close'] >= d]
        rows.append({
            'days_before_close': d,
            'cumulative': len(arrived),
            'fraction_arrived': len(arrived) / total,
            'fraction_remaining': 1 - len(arrived) / total,
        })
    return pd.DataFrame(rows)


def cross_movie_arrival_table(day_buckets=None) -> pd.DataFrame:
    """Build the cross-movie arrival curve: percentiles of fraction_remaining at each day.
    
    This is the main input for a cross-movie lambda estimator.
    
    IMPORTANT: 98% of timestamps are day-level (resolved to 00:00 UTC).
    This table is reliable at daily resolution only. Sub-day estimates
    are meaningless for most movies.
    """
    if day_buckets is None:
        day_buckets = list(range(0, 22))
    
    all_curves = {}
    for slug in reviews['movie_slug'].unique():
        mv = reviews[reviews['movie_slug'] == slug]
        if len(mv) < 10:
            continue
        total = len(mv)
        curve = {}
        for d in day_buckets:
            arrived = len(mv[mv['days_before_close'] >= d])
            curve[d] = 1 - arrived / total  # fraction remaining
        all_curves[slug] = curve
    
    # Build percentile table
    rows = []
    for d in day_buckets:
        fracs = [all_curves[s][d] for s in all_curves]
        rows.append({
            'days_before_close': d,
            'p10': np.percentile(fracs, 10),
            'p25': np.percentile(fracs, 25),
            'median': np.median(fracs),
            'p75': np.percentile(fracs, 75),
            'p90': np.percentile(fracs, 90),
            'mean': np.mean(fracs),
            'n_movies': len(fracs),
        })
    return pd.DataFrame(rows)

arrival_table = cross_movie_arrival_table()
arrival_table

In [ ]:
def load_price_history(slug: str, resolution: str = 'hour') -> pd.DataFrame:
    """Load Kalshi price history for a movie.
    
    Args:
        slug: movie slug
        resolution: 'minute', 'hour', or 'day'
    
    Returns DataFrame with timestamp index and one column per threshold (e.g. 'Above 75').
    Prices are in cents (0-100). NaN = no trade at that timestamp.
    """
    price_dir = ROOT / 'rt-price-histories' / slug
    if not price_dir.exists():
        raise FileNotFoundError(f"No price history directory for {slug}")
    
    files = list(price_dir.glob(f'*-{resolution}.csv'))
    if not files:
        raise FileNotFoundError(f"No {resolution}-level price file for {slug}")
    
    df = pd.read_csv(files[0])
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df = df.set_index('timestamp').sort_index()
    return df

# Quick test: load one
try:
    ph = load_price_history('forbidden_fruits_2026', 'hour')
    print(f"forbidden_fruits price history: {len(ph)} rows, thresholds: {list(ph.columns)}")
    # Show which thresholds have data
    print(f"Non-null prices per threshold: {ph.notna().sum().to_dict()}")
except FileNotFoundError as e:
    print(e)

---

## 3. Lambda exploration

The cross-movie arrival table above gives fraction_remaining at daily resolution.
98% of timestamps are day-level (00:00 UTC), so sub-day is unreliable from this dataset.

**Known facts:**
- Median ~7% of reviews arrive on close day (day 0), ~3% on day -1, ~21% on day -2 (embargo burst)
- 6 movies have minute-level data: the_drama (145m), the_super_mario_galaxy_movie (162m),
  they_will_kill_you (42m), forbidden_fruits_2026 (17m), project_hail_mary (25m),
  ready_or_not_2_here_i_come (23m) — but minute-level is mostly from embargo lift, not close-day

**Directions to explore:**
- Can fraction_remaining predict actual mu = additional reviews better than the naive 6h rate?
- Does conditioning on current review count improve the estimate?
- What does the within-day arrival pattern look like for the 6 minute-level movies?

In [ ]:
# Cross-movie arrival curve plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(arrival_table['days_before_close'], arrival_table['p25'], arrival_table['p75'], alpha=0.2, label='P25–P75')
ax.fill_between(arrival_table['days_before_close'], arrival_table['p10'], arrival_table['p90'], alpha=0.1, label='P10–P90')
ax.plot(arrival_table['days_before_close'], arrival_table['median'], 'k-', lw=2, label='Median')
ax.set_xlabel('Days before close')
ax.set_ylabel('Fraction of reviews remaining')
ax.set_title('Cross-movie review arrival curve (141 movies, daily resolution)')
ax.legend()
ax.set_xlim(0, 21)
ax.invert_xaxis()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()

In [ ]:
# Scatter: actual additional reviews vs what the arrival curve would predict
# at T = 1 day before close (the most operationally relevant window)
snap_24h = snapshot_all_movies(24)
snap_24h['predicted_additional'] = snap_24h['total'] * arrival_table.loc[arrival_table['days_before_close'] == 1, 'median'].values[0] / (1 - arrival_table.loc[arrival_table['days_before_close'] == 1, 'median'].values[0])

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(snap_24h['predicted_additional'], snap_24h['future_reviews'], alpha=0.5, s=20)
mx = max(snap_24h['predicted_additional'].max(), snap_24h['future_reviews'].max()) * 1.05
ax.plot([0, mx], [0, mx], 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('Predicted additional reviews (from arrival curve)')
ax.set_ylabel('Actual additional reviews')
ax.set_title('Cross-movie arrival curve prediction vs reality (T-24h)')
ax.legend()
plt.tight_layout()

In [ ]:
# Your lambda exploration here


---

## 4. p_fresh exploration

**Known facts (from dataset survey):**
- Top critics are ~6pp more negative than non-top (median across 141 movies)
- Top critics are ~49% of the first 20% of reviews, ~16% of the remaining 80%
- This means the naive running average is biased low early in lifecycle (when top critics dominate)
- Cross-movie fresh rate: ~70% overall, wide range (20%–100%)

**Directions to explore:**
- How much does the top-critic correction actually move p_fresh?
- Is there a cross-movie prior (shrinkage) that helps for low-review-count movies?
- Does p_fresh drift systematically over a movie's lifecycle?
- Does the gap between top and non-top freshness vary with movie quality?

In [ ]:
# Top-critic bias across 141 movies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of top-critic bias (pp)
axes[0].hist(movie_stats['top_bias_pp'].dropna(), bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', ls='--', alpha=0.5)
axes[0].axvline(movie_stats['top_bias_pp'].median(), color='blue', ls='--', label=f"Median: {movie_stats['top_bias_pp'].median():.1f}pp")
axes[0].set_xlabel('Top-critic bias (pp): top_fresh_rate - nontop_fresh_rate')
axes[0].set_ylabel('Movies')
axes[0].set_title('Top-critic freshness bias')
axes[0].legend()

# Right: top-critic fraction over lifecycle (how it changes from early to late reviews)
early_top_fracs = []
late_top_fracs = []
for slug in reviews['movie_slug'].unique():
    mv = reviews[reviews['movie_slug'] == slug].sort_values('estimated_timestamp')
    n = len(mv)
    if n < 20:
        continue
    cutoff = int(n * 0.2)
    early_top_fracs.append(mv.iloc[:cutoff]['top_critic'].mean())
    late_top_fracs.append(mv.iloc[cutoff:]['top_critic'].mean())

axes[1].scatter(early_top_fracs, late_top_fracs, alpha=0.5, s=20)
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('Top-critic fraction (first 20% of reviews)')
axes[1].set_ylabel('Top-critic fraction (remaining 80%)')
axes[1].set_title('Top-critic concentration: early vs late')
plt.tight_layout()

In [ ]:
# Naive p_fresh vs actual future fresh rate — how wrong is it?
# At T-24h and T-48h across all 141 movies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, T in zip(axes, [24, 48]):
    snap = snapshot_all_movies(T)
    snap = snap[snap['future_reviews'] >= 3]  # need some future reviews to measure
    
    ax.scatter(snap['p_fresh_naive'], snap['future_fresh_rate'], alpha=0.5, s=20)
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='y=x (perfect)')
    
    # Error stats
    err = snap['future_fresh_rate'] - snap['p_fresh_naive']
    ax.set_xlabel('Naive p_fresh (fresh/total at time T)')
    ax.set_ylabel('Actual future fresh rate')
    ax.set_title(f'T-{T}h: naive vs actual (n={len(snap)}, median err={err.median()*100:.1f}pp)')
    ax.legend()

plt.tight_layout()

In [ ]:
# Your p_fresh exploration here


---

## 5. Edge timing analysis

For movies with both minute-level timestamps near close AND price histories, compute
the naive model's edge at each time point and find when |edge| is maximized.

**Available movies:**
- forbidden_fruits_2026: 8 m/h reviews in last 48h + price history
- they_will_kill_you: 8 m/h reviews in last 48h + price history
- project_hail_mary: price history, but only day-level timestamps near close
- ready_or_not_2_here_i_come: price history, but only day-level timestamps near close

(the_drama and the_super_mario_galaxy_movie have minute-level data but no resolved markets / no price histories)

In [ ]:
def edge_trajectory(slug: str, threshold: int, 
                    hours_range: tuple = (1, 72), step_hours: float = 1,
                    lambda_fn=None, p_fresh_fn=None) -> pd.DataFrame:
    """Compute edge vs time-to-close for one movie/threshold.
    
    At each time step, reconstructs the review state, gets the market price,
    and calls compute_edge.
    
    Args:
        slug: movie slug
        threshold: Kalshi threshold (e.g. 75 for 'Above 75')
        hours_range: (min, max) hours before close to evaluate
        step_hours: step size in hours
        lambda_fn: callable(snapshot_dict, hours_to_close) -> lambda_rate.
                   If None, uses naive (reviews in last 6h / 6).
        p_fresh_fn: callable(snapshot_dict) -> p_fresh.
                    If None, uses naive (fresh/total).
    
    Returns DataFrame with one row per time step.
    """
    close_date = mi.loc[mi['Slug'] == slug, 'Bet Close Date'].values[0]
    close_ts = pd.Timestamp(close_date)
    
    # Load price history (hour-level, forward-fill for interpolation)
    try:
        prices = load_price_history(slug, 'hour')
    except FileNotFoundError:
        prices = load_price_history(slug, 'day')
    
    threshold_col = f'Above {threshold}'
    if threshold_col not in prices.columns:
        raise ValueError(f"No price data for {threshold_col} in {slug}")
    
    prices_filled = prices[threshold_col].ffill()
    
    mv = reviews[reviews['movie_slug'] == slug].copy()
    
    rows = []
    h_min, h_max = hours_range
    for hours_to_close in np.arange(h_min, h_max + step_hours, step_hours):
        snap = snapshot_at(slug, hours_to_close)
        
        if snap['total'] == 0:
            continue
        
        # Get market price at this time
        eval_time = close_ts - pd.Timedelta(hours=hours_to_close)
        price_idx = prices_filled.index.searchsorted(eval_time)
        if price_idx == 0:
            continue
        market_price = prices_filled.iloc[price_idx - 1]
        if pd.isna(market_price):
            continue
        
        # Compute parameters
        if lambda_fn is not None:
            lam = lambda_fn(snap, hours_to_close)
        else:
            # Naive: count reviews in the 6h window before this eval time
            window_start = eval_time - pd.Timedelta(hours=6)
            recent = mv[(mv['estimated_timestamp'] >= window_start) & 
                        (mv['estimated_timestamp'] < eval_time)]
            lam = len(recent) / 6.0
        
        if p_fresh_fn is not None:
            pf = p_fresh_fn(snap)
        else:
            pf = snap['p_fresh_naive']
        
        result = compute_edge(
            threshold=threshold,
            market_price=market_price,
            fresh_count=snap['fresh'],
            total_count=snap['total'],
            hours_to_close=hours_to_close,
            lambda_rate=lam,
            p_fresh=pf,
        )
        
        rows.append({
            'hours_to_close': hours_to_close,
            'eval_time': eval_time,
            'market_price': market_price,
            'fresh': snap['fresh'],
            'total': snap['total'],
            'lambda_rate': lam,
            'p_fresh': pf,
            **result,
        })
    
    return pd.DataFrame(rows)

In [ ]:
# Example: edge trajectory for forbidden_fruits, threshold near its final score
# Final score range: 75.5%-80.5%, so "Above 75" is the interesting threshold
# Swap slug/threshold as needed

slug = 'forbidden_fruits_2026'
threshold = 75

traj = edge_trajectory(slug, threshold, hours_range=(1, 72), step_hours=1)
if not traj.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    axes[0].plot(traj['hours_to_close'], traj['edge_cents'], 'b-', lw=1.5)
    axes[0].axhline(0, color='gray', ls='--', alpha=0.5)
    axes[0].fill_between(traj['hours_to_close'], traj['edge_cents'], 0, alpha=0.15)
    max_edge_idx = traj['edge_cents'].abs().idxmax()
    max_row = traj.loc[max_edge_idx]
    axes[0].axvline(max_row['hours_to_close'], color='red', ls=':', alpha=0.7,
                     label=f"Max |edge| = {max_row['edge_cents']:.1f}c at T-{max_row['hours_to_close']:.0f}h")
    axes[0].set_ylabel('Edge (cents)')
    axes[0].set_title(f'{slug} — Above {threshold} — Edge trajectory (naive lambda + naive p_fresh)')
    axes[0].legend()
    
    axes[1].plot(traj['hours_to_close'], traj['market_price'], 'g-', lw=1.5, label='Market price')
    axes[1].plot(traj['hours_to_close'], traj['p_yes'] * 100, 'b--', lw=1.5, label='Model P(Yes)')
    axes[1].set_xlabel('Hours to close')
    axes[1].set_ylabel('Cents / Probability %')
    axes[1].legend()
    axes[1].invert_xaxis()
    
    plt.tight_layout()
else:
    print(f"No data for {slug} / Above {threshold} — try a different threshold")

In [ ]:
# Your edge timing exploration here
# 
# To compare a custom estimator against naive, pass lambda_fn and/or p_fresh_fn:
#
#   def my_lambda(snap, hours_to_close):
#       # snap has: fresh, total, top_fresh, top_total, nontop_fresh, nontop_total, future_reviews, ...
#       return ...
#
#   def my_p_fresh(snap):
#       return ...
#
#   traj_custom = edge_trajectory('forbidden_fruits_2026', 75, lambda_fn=my_lambda, p_fresh_fn=my_p_fresh)


---

## 6. Plug your estimators into compute_edge

Once you've built estimator functions, test them end-to-end here. The `edge_trajectory`
function above accepts `lambda_fn` and `p_fresh_fn` callables for A/B comparison.

When ready, promote the functions to `edge.py` (or a new `estimators.py`) to replace
the naive defaults.

In [ ]:
# Example: test a custom estimator against naive on one movie

# def my_lambda(snap, hours_to_close):
#     ...

# def my_p_fresh(snap):
#     ...

# slug, threshold = 'forbidden_fruits_2026', 75
# traj_naive = edge_trajectory(slug, threshold)
# traj_custom = edge_trajectory(slug, threshold, lambda_fn=my_lambda, p_fresh_fn=my_p_fresh)

# fig, ax = plt.subplots(figsize=(12, 5))
# ax.plot(traj_naive['hours_to_close'], traj_naive['edge_cents'], 'b-', label='Naive')
# ax.plot(traj_custom['hours_to_close'], traj_custom['edge_cents'], 'r-', label='Custom')
# ax.axhline(0, color='gray', ls='--', alpha=0.5)
# ax.set_xlabel('Hours to close')
# ax.set_ylabel('Edge (cents)')
# ax.set_title(f'{slug} — Above {threshold}')
# ax.legend()
# ax.invert_xaxis()
# plt.tight_layout()
